# Predição de Risco de Acidentes

Este notebook tem como objetivo criar um modelo preditivo para o risco de acidentes (`baixo`, `medio`, `alto`), com foco na classe de alto risco.

## Passos:
1. Carregamento e Preparação dos Dados
2. Separação de Dados de Validação (20%)
3. Pipelines de Pré-processamento
4. Tunning de Hiperparâmetros com Optuna (Logistic Regression vs LightGBM)
5. Avaliação dos Modelos (Separada)

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, accuracy_score, make_scorer, roc_curve, auc, precision_recall_curve
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.INFO)
warnings.filterwarnings('ignore')

## 1. Carregamento e Preparação dos Dados

In [ ]:
# Carregar dados com Polars
df_pl = pl.read_parquet("data/anuario_prf.parquet")

# Converter para Pandas
df = df_pl.to_pandas()

# Visualizar primeiras linhas
df.head()

In [ ]:
# Verificar tipos e nulos
df.info()

## 2. Separação de Dados de Validação

In [ ]:
TARGET = 'risco'

# Remover colunas que não devem ser usadas na predição (se houver)
# A coluna 'data' é do tipo object/datetime, vamos removê-la pois já temos 'mes' e 'dia_semana_num'
if 'data' in df.columns:
    df = df.drop(columns=['data'])

# Separar X e y
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Codificar o target para números para facilitar o LightGBM e métricas
# Ordem: baixo, medio, alto (assumindo ordem de gravidade)
risk_mapping = {'baixo': 0, 'medio': 1, 'alto': 2}
y = y.map(risk_mapping)

# Separar 20% para validação final (Stratified para manter proporção das classes)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Salvar conjunto de validação em parquet
val_df = pd.concat([X_val, y_val], axis=1)
pl.from_pandas(val_df).write_parquet("data/validação.parquet")

print(f"Treino shape: {X_train.shape}")
print(f"Validação shape: {X_val.shape}")
print("Distribuição classes treino:")
print(y_train.value_counts(normalize=True))
print("Distribuição classes validação:")
print(y_val.value_counts(normalize=True))

## 3. Definição de Pipelines e Pré-processamento

In [ ]:
# Identificar colunas numéricas e categóricas
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['number']).columns.tolist()

print("Numéricas:", numerical_cols)
print("Categóricas:", categorical_cols)

# Preprocessamento para Regressão Logística
numeric_transformer_lr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_lr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_lr, numerical_cols),
        ('cat', categorical_transformer_lr, categorical_cols)
    ])

# Preprocessamento para LightGBM
numeric_transformer_lgbm = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()) 
])

categorical_transformer_lgbm = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor_lgbm = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_lgbm, numerical_cols),
        ('cat', categorical_transformer_lgbm, categorical_cols)
    ])

## 4. Tunning de Hiperparâmetros com Optuna

In [19]:
def apply_threshold(y_proba, threshold):
    # y_proba shape: (n_samples, 3) -> [baixo, medio, alto]
    # Se prob(alto) >= threshold, prediz alto (2)
    # Caso contrário, prediz o argmax entre baixo (0) e medio (1)
    
    preds = np.argmax(y_proba[:, :2], axis=1) # Prediz 0 ou 1 baseado no maior entre eles
    
    # Onde a probabilidade de alto for maior que o threshold, sobrescreve com 2
    mask_alto = y_proba[:, 2] >= threshold
    preds[mask_alto] = 2
    
    return preds

def objective_lr(trial):
    # Hiperparâmetros
    C = trial.suggest_loguniform('C', 1e-3, 100)
    # Fixando solver e class_weight como pedido, mas usando lista para suggest_categorical
    solver = trial.suggest_categorical('solver', ['lbfgs'])
    class_weight = trial.suggest_categorical('class_weight', ['balanced'])
    
    # Limiar para classe 'alto'
    threshold = trial.suggest_float('threshold', 0.2, 0.9, step=0.05)
    
    model = LogisticRegression(
        C=C, 
        solver=solver, 
        class_weight=class_weight, 
        max_iter=100, 
        random_state=42
    )
    
    clf = Pipeline(steps=[('preprocessor', preprocessor_lr),
                          ('classifier', model)])
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    f1_scores_alto = []
    
    # Manual Cross-Validation para aplicar threshold customizado
    for train_index, test_index in skf.split(X_train, y_train):
        X_tr, X_te = X_train.iloc[train_index], X_train.iloc[test_index]
        y_tr, y_te = y_train.iloc[train_index], y_train.iloc[test_index]
        
        clf.fit(X_tr, y_tr)
        y_proba = clf.predict_proba(X_te)
        
        y_pred = apply_threshold(y_proba, threshold)
        
        # Calcular F1 apenas da classe 'alto' (classe 2)
        # average=None retorna array [f1_0, f1_1, f1_2]
        f1_per_class = f1_score(y_te, y_pred, average=None)
        if len(f1_per_class) > 2:
            f1_scores_alto.append(f1_per_class[2])
        else:
            # Caso raro onde a classe 2 não foi predita nenhuma vez ou não existe no fold
            f1_scores_alto.append(0.0)
            
    mean_f1_alto = np.mean(f1_scores_alto)
    print(f"Trial {trial.number}: F1 Alto = {mean_f1_alto:.4f} (Thresh: {threshold:.2f})")
    return mean_f1_alto

def objective_lgbm(trial):
    # Hiperparâmetros
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 3000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 500),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
    }
    
    model = lgb.LGBMClassifier(**params)
    
    clf = Pipeline(steps=[('preprocessor', preprocessor_lgbm),
                          ('classifier', model)])
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_val_score(clf, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
    
    mean_score = scores.mean()
    print(f"Trial {trial.number}: F1 Macro = {mean_score:.4f}")
    return mean_score

In [20]:
# Executar estudo Logistic Regression
print("Iniciando estudo Logistic Regression (Otimizando F1 da classe Alto)...")
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=50) 
print("Melhores parâmetros LR:", study_lr.best_params)
print("Melhor F1 Alto LR:", study_lr.best_value)

[I 2025-11-24 18:44:25,882] A new study created in memory with name: no-name-282e8926-6dc7-4e3f-b16e-5d300f3b333b


Iniciando estudo Logistic Regression (Otimizando F1 da classe Alto)...


[W 2025-11-24 18:44:48,114] Trial 0 failed with parameters: {'C': 0.17571117215735044, 'solver': 'lbfgs', 'class_weight': 'balanced', 'threshold': 0.55} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/aderson/miniconda3/envs/dengue/lib/python3.12/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_190709/1363572253.py", line 44, in objective_lr
    clf.fit(X_tr, y_tr)
  File "/home/aderson/miniconda3/envs/dengue/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aderson/miniconda3/envs/dengue/lib/python3.12/site-packages/sklearn/pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/home/aderson/miniconda3/envs/dengue/lib/python3.12/site-packages/sklea

KeyboardInterrupt: 

In [ ]:
# Executar estudo LightGBM
print("Iniciando estudo LightGBM...")
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=20)
print("Melhores parâmetros LGBM:", study_lgbm.best_params)
print("Melhor F1 Macro LGBM:", study_lgbm.best_value)

## 5. Avaliação dos Modelos

In [ ]:
def plot_roc_curve(y_test, y_score, n_classes, title='ROC Curve'):
    from sklearn.preprocessing import label_binarize
    from sklearn.metrics import roc_curve, auc
    
    # Binarize the output
    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    
    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Plot all ROC curves
    plt.figure(figsize=(8, 6))
    colors = ['blue', 'red', 'green']
    classes = ['baixo', 'medio', 'alto']
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label='ROC curve of class {0} (area = {1:0.2f})'
                 ''.format(classes[i], roc_auc[i]))

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Chance')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.show()

def evaluate_model(model, X_train, y_train, X_val, y_val, model_name, threshold=None):
    print(f"--- Avaliação: {model_name} ---")
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)
    
    if threshold is not None:
        print(f"Aplicando threshold customizado: {threshold} para classe 'alto'")
        y_pred = apply_threshold(y_pred_proba, threshold)
    else:
        y_pred = model.predict(X_val)
    
    print("\nRelatório de Classificação:")
    print(classification_report(y_val, y_pred, target_names=['baixo', 'medio', 'alto']))
    
    print("\nMatriz de Confusão:")
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['baixo', 'medio', 'alto'], yticklabels=['baixo', 'medio', 'alto'])
    plt.title(f'Matriz de Confusão - {model_name}')
    plt.ylabel('Real')
    plt.xlabel('Predito')
    plt.show()
    
    print("\nCurva ROC:")
    plot_roc_curve(y_val, y_pred_proba, n_classes=3, title=f'ROC Curve - {model_name}')
    
    if threshold is not None:
        # Plotar F1 vs Threshold para classe Alto
        thresholds = np.arange(0.2, 0.95, 0.05)
        f1_scores = []
        for t in thresholds:
            p = apply_threshold(y_pred_proba, t)
            f1 = f1_score(y_val, p, average=None)[2]
            f1_scores.append(f1)
            
        plt.figure(figsize=(8, 4))
        plt.plot(thresholds, f1_scores, label='F1 Class Alto')
        plt.axvline(threshold, color='r', linestyle='--', label=f'Selected Threshold ({threshold:.2f})')
        plt.xlabel('Threshold')
        plt.ylabel('F1 Score (Alto)')
        plt.title('F1 Score vs Threshold (Validation Set)')
        plt.legend()
        plt.grid(True)
        plt.show()

### 5.1 Logistic Regression

In [ ]:
print("Recuperando melhores parâmetros Logistic Regression...")
best_params_lr = study_lr.best_params.copy()
best_threshold_lr = best_params_lr.pop('threshold') # Remover threshold dos params do modelo

model_lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    **best_params_lr
)
pipeline_lr = Pipeline(steps=[('preprocessor', preprocessor_lr),
                              ('classifier', model_lr)])

evaluate_model(pipeline_lr, X_train, y_train, X_val, y_val, "Logistic Regression", threshold=best_threshold_lr)

### 5.2 LightGBM

In [ ]:
print("Recuperando melhores parâmetros LightGBM...")
best_params_lgbm = study_lgbm.best_params
model_lgbm = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric='multi_logloss',
    verbosity=-1,
    boosting_type='gbdt',
    random_state=42,
    **best_params_lgbm
)
pipeline_lgbm = Pipeline(steps=[('preprocessor', preprocessor_lgbm),
                                ('classifier', model_lgbm)])

evaluate_model(pipeline_lgbm, X_train, y_train, X_val, y_val, "LightGBM")